In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split 

In [ ]:
df = pd.read_csv("loan_approval_data.csv")

df.isnull().sum() # all column have 50 null values
df.head()
df.describe()

## Data Cleaning - Handling missing values 
##### for numeric values we fill mean value in place of null values
##### for categorical values we fill mode(most repeated) values in place of null values

In [ ]:
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = ["number"]).columns

In [ ]:
numerical_cols

In [ ]:
from sklearn.impute import SimpleImputer

# Filling numerical cols 

num_imp = SimpleImputer(strategy = "mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])



In [ ]:
# Filling categorical cols

cat_imp = SimpleImputer(strategy = "most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

In [ ]:
df.info()

## EDA

In [ ]:
# how balanced our classes are 

classes_count = df["Loan_Approved"].value_counts()
print(classes_count)

plt.pie(classes_count,autopct = "%1.1f%%",wedgeprops = {
            "edgecolor": "black",
            "linewidth": 1.5})
plt.title("Loan approved or not")

In [ ]:
# Gender split in data 

gender_count = df["Gender"].value_counts()
print(gender_count)
ax = sns.barplot(gender_count)
ax.bar_label(ax.containers[0])

# Education level

edu_count = df["Education_Level"].value_counts()
print(edu_count)
ax = sns.barplot(edu_count)
ax.bar_label(ax.containers[1])

plt.xlabel("Gender and Education")

In [ ]:
# Analyze loan approval with income 

sns.histplot(
    data = df,
    x = "Applicant_Income",
    bins = 25
)
sns.histplot(
    data = df,
    x = "Coapplicant_Income",
    bins = 25,
    alpha = 0.3
)

In [ ]:
# To detect outlier we use box plots 

sns.boxplot(
    data = df,
    x = "Loan_Approved",
    y = "Applicant_Income"
)

In [ ]:
# Loan approval with different features 

fig,axes = plt.subplots(2,3)

sns.boxplot(ax = axes[0,0],data = df,x = "Loan_Approved",y = "Applicant_Income")
sns.boxplot(ax = axes[0,1],data = df,x = "Loan_Approved",y = "Credit_Score")
sns.boxplot(ax = axes[0,2],data = df,x = "Loan_Approved",y = "Age")
sns.boxplot(ax = axes[1,0],data = df,x = "Loan_Approved",y = "DTI_Ratio")
sns.boxplot(ax = axes[1,1],data = df,x = "Loan_Approved",y = "Savings")
sns.boxplot(ax = axes[1,2],data = df,x = "Loan_Approved",y = "Loan_Amount")
plt.tight_layout()

In [ ]:
# Credit Score with Loan approved

sns.histplot(
    data = df,
    x = "Credit_Score",
    hue = "Loan_Approved",
    multiple = "dodge"
)

In [ ]:
# Applicant  with Loan approved

sns.histplot(
    data = df,
    x = "Applicant_Income",
    hue = "Loan_Approved",
    multiple = "dodge"
)

In [ ]:
# removing applicant id bcz it doesn't affect the loan approval probability

df = df.drop("Applicant_ID", axis = 1)

df.head()

## Encoding 

In [ ]:
# LabelEncoder = assigns an integer to each category (same as map function)

# OneHotEncoder = creates binary columns for each category (same as get_dummy function)

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# LabelEncoder
le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])

In [ ]:
df.head()

In [ ]:
# OneHotEncoder
cols = ["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

ohe = OneHotEncoder(
    drop = "first",
    sparse_output = False,
    handle_unknown = "ignore"
)

encoded = ohe.fit_transform(df[cols])
# now here it returns the encoded value only not put in data so we have to do it.

encoded # encoded values 
ohe.get_feature_names_out(cols)

encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(cols), index = df.index)

In [ ]:
encoded_df.head() # Now this is the new dataframe we got 

In [ ]:
# To put these values in the original df we will concatenate both df and encoded_df

df = pd.concat([df.drop(columns = cols),encoded_df],axis = 1)

In [ ]:
df.head()
df.info()
# all of the columns are now in number thus the model can understand the data better.

## Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include = "number")
corr_m = num_cols.corr()

In [ ]:
corr_matrix = num_cols.corr()["Loan_Approved"].sort_values(ascending = False)
# tells the relation of every feature with Loan_approved(output)

In [ ]:
plt.figure(figsize = (15,8))
sns.heatmap(
    corr_m,
    annot = True,
    fmt = ".2f",
    cmap = "coolwarm"
)

plt.tight_layout()

# Model training 

## Train-Test-Split + Feature Scaling

In [ ]:
X = df.drop("Loan_Approved",axis = 1)
y = df["Loan_Approved"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size = 0.2,random_state = 42
)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Prediction 

In [ ]:
# Logistic Regression 

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation 
# here for loan based model we need precison_score as the evaluation matrics. 
# as we try to reduce type1 error in which FP occur
from sklearn.metrics import accuracy_score,recall_score,precision_score,confusion_matrix, f1_score

print("Precision :", precision_score(y_test,y_pred)) # most imp here 
print("Recall :", recall_score(y_test,y_pred))
print("Accuracy :", accuracy_score(y_test,y_pred))
print("F1 score :", f1_score(y_test,y_pred))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred))

In [ ]:
# kNN 

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors = 13)
knn_model.fit(X_train_scaled,y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation 
# here for loan based model we need precison_score as the evaluation matrics. 
# as we try to reduce type1 error in which FP occur
from sklearn.metrics import accuracy_score,recall_score,precision_score,confusion_matrix, f1_score

print("Precision :", precision_score(y_test,y_pred)) # most imp here 
print("Recall :", recall_score(y_test,y_pred))
print("Accuracy :", accuracy_score(y_test,y_pred))
print("F1 score :", f1_score(y_test,y_pred))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred))


# Now this model won't perofrm well on this data bcz it has high no. of features 
# and kNN dosn't perform well for such datasets 

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled,y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation 
# here for loan based model we need precison_score as the evaluation matrics. 
# as we try to reduce type1 error in which FP occur
from sklearn.metrics import accuracy_score,recall_score,precision_score,confusion_matrix, f1_score

print("Precision :", precision_score(y_test,y_pred)) # most imp here 
print("Recall :", recall_score(y_test,y_pred))
print("Accuracy :", accuracy_score(y_test,y_pred))
print("F1 score :", f1_score(y_test,y_pred))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred))

### Best model on the basis of precision => Naive Bayes

# Feature Engineering 

In [ ]:
# Adding New Features

df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

# Compressing Skewed Data 

# df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

X = df.drop(columns = ["Loan_Approved","Credit_Score","DTI_Ratio"])
y = df["Loan_Approved"]

# Train_test_split 

X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

# Scaling 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression

log_model = LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred1 = log_model.predict(X_test_scaled)
print("Logistice regression")
print("Precision :", precision_score(y_test,y_pred1)) # most imp here 
print("Recall :", recall_score(y_test,y_pred1))
print("Accuracy :", accuracy_score(y_test,y_pred1))
print("F1 score :", f1_score(y_test,y_pred1))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred1))
print("\n")

# kNN

knn_model = KNeighborsClassifier(n_neighbors = 13)
knn_model.fit(X_train_scaled,y_train)

y_pred = knn_model.predict(X_test_scaled)
print("kNN Model")
print("Precision :", precision_score(y_test,y_pred)) # most imp here 
print("Recall :", recall_score(y_test,y_pred))
print("Accuracy :", accuracy_score(y_test,y_pred))
print("F1 score :", f1_score(y_test,y_pred))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred))
print("\n")

# Naive Bayes

nb_model = GaussianNB()
nb_model.fit(X_train_scaled,y_train)

y_pred = nb_model.predict(X_test_scaled)
print("Naive bayes")
print("Precision :", precision_score(y_test,y_pred)) # most imp here 
print("Recall :", recall_score(y_test,y_pred))
print("Accuracy :", accuracy_score(y_test,y_pred))
print("F1 score :", f1_score(y_test,y_pred))
print("Confusion Matrix :\n" , confusion_matrix(y_test,y_pred))

In [ ]:
results = pd.DataFrame({
    "Loan_ID": X_test.index,
    "Loan_Status": ["Y" if p == 1 else "N" for p in y_pred1]
})

results.to_csv("Solutions.csv", index=False)